# VRTX benchmark on a Kaggle T4

Settings needed (right sidebar): **Accelerator = GPU T4 x2** (or T4) and **Internet = On**.

Then Run All. Takes roughly 15 to 25 minutes. Copy the JSON printed by the last cell.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code

In [ ]:
%cd /kaggle/working
!rm -rf vrtx
!git clone --depth 1 https://github.com/JacobBhatt12/vrtx.git
%cd vrtx
!git log --oneline -1

## 3. Install requirements

Kaggle already ships CUDA PyTorch, so pip keeps it and only adds what is missing.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
assert torch.cuda.is_available(), 'No GPU: set Accelerator to GPU T4 in the sidebar, then restart'
print('GPU:', torch.cuda.get_device_name(0))

## 4. Download Qwen2.5-0.5B-Instruct

In [ ]:
from huggingface_hub import snapshot_download
path = snapshot_download('Qwen/Qwen2.5-0.5B-Instruct', local_dir='models/qwen2.5-0.5b',
                         allow_patterns=['*.json', '*.safetensors', '*.txt'])
!ls -lh models/qwen2.5-0.5b

## 5. Run the tests

Includes `test_fp16_on_gpu_matches_fp32_cpu`, which checks the CUDA + fp16 path.

In [ ]:
!python -m pytest tests/ -v

## 6. Benchmark: continuous batching vs batch 1 vs naive Hugging Face (fp16 on cuda)

In [ ]:
!python -m benchmarks.bench_batching --requests 64 --max-tokens 128 --batches 1,8,32,64 --hf-baseline

## 7. Results JSON (copy everything below and paste it back)

In [ ]:
import json, glob, os
latest = max(glob.glob('benchmarks/results/batching_cuda_*.json'), key=os.path.getmtime)
print(latest)
print(json.dumps(json.load(open(latest)), indent=2))